# Graficos finales para Hawsem.state

Este notebook crea una interfaz con dos pestanas:

- `Histogramas de frecuencia de carga`: usa un archivo `Hawsem.state` y grafica la frecuencia observada de cada estado de carga por residuo.
- `Grafico de pKa`: recorre un directorio con varios `Hawsem.state`, calcula la carga media por residuo para cada pH y estima el pKa por interpolacion.


In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except ImportError:
    widgets = None
    WIDGETS_AVAILABLE = False


def parse_hawsem_state(state_path):
    state_path = Path(state_path).expanduser()
    if not state_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {state_path}")

    snapshots = []
    for line_number, raw_line in enumerate(state_path.read_text().splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue

        end_of_states = line.rfind("]")
        if end_of_states == -1:
            raise ValueError(
                f"Linea {line_number} invalida en {state_path}: no se encontro la lista de estados"
            )

        state_data = json.loads(line[: end_of_states + 1])
        ph_text = line[end_of_states + 1 :].strip()
        if not ph_text:
            raise ValueError(f"Linea {line_number} invalida en {state_path}: falta el pH")

        charges = {int(residue): float(charge) for residue, charge in state_data}
        snapshots.append({"pH": float(ph_text), "charges": charges})

    if not snapshots:
        raise ValueError(f"El archivo {state_path} no contiene datos utiles")

    return snapshots


def state_file_to_dataframe(state_path):
    snapshots = parse_hawsem_state(state_path)
    residues = sorted({residue for snapshot in snapshots for residue in snapshot["charges"]})

    records = []
    for frame_id, snapshot in enumerate(snapshots, start=1):
        for residue in residues:
            records.append(
                {
                    "frame": frame_id,
                    "pH": snapshot["pH"],
                    "residue": residue,
                    "charge": snapshot["charges"].get(residue, 0.0),
                }
            )

    return pd.DataFrame(records), residues


def summarize_directory(directory, filename="Hawsem.state"):
    directory = Path(directory).expanduser()
    if not directory.exists():
        raise FileNotFoundError(f"No existe el directorio: {directory}")

    state_files = sorted(directory.rglob(filename))
    if not state_files:
        raise FileNotFoundError(
            f"No se encontraron archivos {filename} dentro de {directory}"
        )

    records = []
    residues = set()
    warnings = []

    for state_file in state_files:
        snapshots = parse_hawsem_state(state_file)
        ph_values = sorted({snapshot["pH"] for snapshot in snapshots})
        if len(ph_values) != 1:
            warnings.append(
                f"{state_file}: se encontraron varios pH {ph_values}; se usara el promedio"
            )
        ph_value = float(np.mean(ph_values))

        file_residues = sorted(
            {residue for snapshot in snapshots for residue in snapshot["charges"]}
        )
        residues.update(file_residues)

        for residue in file_residues:
            charges = [snapshot["charges"].get(residue, 0.0) for snapshot in snapshots]
            records.append(
                {
                    "file": str(state_file),
                    "pH": ph_value,
                    "residue": residue,
                    "mean_charge": float(np.mean(charges)),
                    "samples": len(charges),
                }
            )

    summary = pd.DataFrame(records).sort_values(["residue", "pH", "file"])
    return summary, sorted(residues), warnings, state_files


def estimate_pka(curve_df):
    curve_df = curve_df.sort_values("pH")
    if len(curve_df) < 2:
        return np.nan

    y_values = curve_df["mean_charge"].to_numpy(dtype=float)
    x_values = curve_df["pH"].to_numpy(dtype=float)

    y_min = np.min(y_values)
    y_max = np.max(y_values)
    if np.isclose(y_min, y_max):
        return np.nan

    target = (y_min + y_max) / 2.0

    for x1, x2, y1, y2 in zip(x_values[:-1], x_values[1:], y_values[:-1], y_values[1:]):
        if np.isclose(y1, target):
            return float(x1)
        if np.isclose(y2, target):
            return float(x2)

        crosses_target = (y1 - target) * (y2 - target) < 0
        if crosses_target and not np.isclose(y1, y2):
            fraction = (target - y1) / (y2 - y1)
            return float(x1 + fraction * (x2 - x1))

    nearest_index = int(np.argmin(np.abs(y_values - target)))
    return float(x_values[nearest_index])


def plot_charge_histograms(df, residues, normalize=True):
    residues = list(residues)
    if not residues:
        raise ValueError("Selecciona al menos un residuo")

    unique_charges = sorted(df["charge"].unique())
    fig, axes = plt.subplots(len(residues), 1, figsize=(8, 4 * len(residues)), squeeze=False)

    for axis, residue in zip(axes.flatten(), residues):
        residue_data = df[df["residue"] == residue]
        counts = residue_data["charge"].value_counts().reindex(unique_charges, fill_value=0).sort_index()
        values = counts / counts.sum() if normalize else counts

        axis.bar([str(charge) for charge in values.index], values.values, color="#1f77b4")
        axis.set_title(f"Residuo {residue}")
        axis.set_xlabel("Carga observada")
        axis.set_ylabel("Frecuencia" if normalize else "Conteo")
        axis.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()


def plot_pka_analysis(summary_df, residues):
    residues = list(residues)
    if not residues:
        raise ValueError("Selecciona al menos un residuo")

    selected = summary_df[summary_df["residue"].isin(residues)].copy()
    if selected.empty:
        raise ValueError("No hay datos para los residuos seleccionados")

    pka_rows = []
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for residue in residues:
        curve = selected[selected["residue"] == residue].sort_values("pH")
        axes[0].plot(curve["pH"], curve["mean_charge"], marker="o", label=f"Residuo {residue}")
        pka_rows.append({"residue": residue, "pKa": estimate_pka(curve)})

    pka_df = pd.DataFrame(pka_rows).dropna().sort_values("residue")

    axes[0].set_title("Curvas de titulacion")
    axes[0].set_xlabel("pH")
    axes[0].set_ylabel("Carga media")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    if pka_df.empty:
        axes[1].text(0.5, 0.5, "No se pudo estimar un pKa", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        axes[1].bar(pka_df["residue"].astype(str), pka_df["pKa"], color="#ff7f0e")
        axes[1].set_title("pKa estimado por residuo")
        axes[1].set_xlabel("Residuo")
        axes[1].set_ylabel("pKa")
        axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

    if not pka_df.empty:
        display(pka_df.reset_index(drop=True))


def guess_default_state_file():
    candidates = [
        Path("Hawsem.state"),
        Path("1ZUG/Hawsem.state"),
        Path("../1ZUG/Hawsem.state"),
        Path("./helpers/Hawsem.state"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return "1ZUG/Hawsem.state"


def guess_default_directory():
    candidates = [Path("."), Path("1ZUG"), Path(".."), Path("../1ZUG")]
    for candidate in candidates:
        if any(candidate.rglob("Hawsem.state")):
            return str(candidate)
    return "."


if WIDGETS_AVAILABLE:
    hist_file = widgets.Text(
    value=guess_default_state_file(),
    description="Archivo:",
    layout=widgets.Layout(width="95%"),
)
    hist_load = widgets.Button(description="Cargar Hawsem.state", button_style="info")
    hist_residues = widgets.SelectMultiple(
        options=[],
        description="Residuos:",
        layout=widgets.Layout(width="95%", height="220px"),
    )
    hist_normalize = widgets.Checkbox(value=True, description="Mostrar frecuencia normalizada")
    hist_plot = widgets.Button(description="Generar histogramas", button_style="primary")
    hist_message = widgets.HTML()
    hist_output = widgets.Output()
    hist_cache = {"df": None, "residues": []}
else:
    hist_cache = {"df": None, "residues": []}


def load_histogram_data(_=None):
    with hist_output:
        hist_output.clear_output()
        try:
            df, residues = state_file_to_dataframe(hist_file.value)
            hist_cache["df"] = df
            hist_cache["residues"] = residues
            hist_residues.options = residues
            hist_residues.value = tuple(residues[: min(4, len(residues))])
            hist_message.value = (
                f"<b>Archivo cargado.</b> {len(df['frame'].unique())} snapshots, "
                f"{len(residues)} residuos titulables y pH {df['pH'].iloc[0]:.2f}."
            )
        except Exception as exc:
            hist_cache["df"] = None
            hist_cache["residues"] = []
            hist_residues.options = []
            hist_residues.value = ()
            hist_message.value = f"<span style='color:#b00020;'><b>Error:</b> {exc}</span>"


def run_histogram_plot(_=None):
    with hist_output:
        hist_output.clear_output()
        try:
            if hist_cache["df"] is None:
                load_histogram_data()
            if hist_cache["df"] is None:
                return
            plot_charge_histograms(
                hist_cache["df"],
                hist_residues.value,
                normalize=hist_normalize.value,
            )
        except Exception as exc:
            print(f"Error al generar histogramas: {exc}")


if WIDGETS_AVAILABLE:
    hist_load.on_click(load_histogram_data)
    hist_plot.on_click(run_histogram_plot)

if WIDGETS_AVAILABLE:
    hist_controls = widgets.VBox([
        widgets.HTML("<h3>Histogramas de frecuencia de carga</h3>"),
        widgets.HTML("<p>Carga un archivo <code>Hawsem.state</code>, elige uno o mas residuos y genera el histograma de cargas observadas.</p>"),
        hist_file,
        widgets.HBox([hist_load, hist_plot]),
        hist_normalize,
        hist_residues,
        hist_message,
        hist_output,
    ])


if WIDGETS_AVAILABLE:
    pka_directory = widgets.Text(
        value=guess_default_directory(),
        description="Directorio:",
        layout=widgets.Layout(width="95%"),
    )
    pka_pattern = widgets.Text(
        value="Hawsem.state",
        description="Nombre:",
        layout=widgets.Layout(width="95%"),
    )
    pka_load = widgets.Button(description="Buscar archivos", button_style="info")
    pka_residues = widgets.SelectMultiple(
        options=[],
        description="Residuos:",
        layout=widgets.Layout(width="95%", height="220px"),
    )
    pka_plot = widgets.Button(description="Generar grafico de pKa", button_style="primary")
    pka_message = widgets.HTML()
    pka_output = widgets.Output()
    pka_cache = {"summary": None, "residues": []}
else:
    pka_cache = {"summary": None, "residues": []}


def load_pka_data(_=None):
    with pka_output:
        pka_output.clear_output()
        try:
            summary, residues, warnings, state_files = summarize_directory(
                pka_directory.value,
                filename=pka_pattern.value,
            )
            pka_cache["summary"] = summary
            pka_cache["residues"] = residues
            pka_residues.options = residues
            pka_residues.value = tuple(residues[: min(4, len(residues))])

            ph_values = sorted(summary["pH"].unique())
            warning_html = ""
            if warnings:
                joined = "<br>".join(warnings)
                warning_html = f"<br><span style='color:#8a6d3b;'><b>Aviso:</b><br>{joined}</span>"

            pka_message.value = (
                f"<b>Analisis cargado.</b> {len(state_files)} archivos encontrados con "
                f"{len(ph_values)} valores de pH y {len(residues)} residuos detectados.{warning_html}"
            )
        except Exception as exc:
            pka_cache["summary"] = None
            pka_cache["residues"] = []
            pka_residues.options = []
            pka_residues.value = ()
            pka_message.value = f"<span style='color:#b00020;'><b>Error:</b> {exc}</span>"


def run_pka_plot(_=None):
    with pka_output:
        pka_output.clear_output()
        try:
            if pka_cache["summary"] is None:
                load_pka_data()
            if pka_cache["summary"] is None:
                return
            plot_pka_analysis(pka_cache["summary"], pka_residues.value)
        except Exception as exc:
            print(f"Error al generar el grafico de pKa: {exc}")


if WIDGETS_AVAILABLE:
    pka_load.on_click(load_pka_data)
    pka_plot.on_click(run_pka_plot)

if WIDGETS_AVAILABLE:
    pka_controls = widgets.VBox([
        widgets.HTML("<h3>Grafico de pKa desde varios Hawsem.state</h3>"),
        widgets.HTML("<p>Indica un directorio, busca archivos <code>Hawsem.state</code> en forma recursiva y estima el pKa a partir de la carga media de cada residuo.</p>"),
        pka_directory,
        pka_pattern,
        widgets.HBox([pka_load, pka_plot]),
        pka_residues,
        pka_message,
        pka_output,
    ])


if WIDGETS_AVAILABLE:
    tabs = widgets.Tab(children=[hist_controls, pka_controls])
    tabs.set_title(0, "Histogramas")
    tabs.set_title(1, "pKa")

    display(tabs)

    load_histogram_data()
    load_pka_data()
else:
    display(
        Markdown(
            "**ipywidgets no esta instalado en este kernel.** Instala `ipywidgets` para usar las pestanas interactivas del notebook."
        )
    )


**ipywidgets no esta instalado en este kernel.** Instala `ipywidgets` para usar las pestanas interactivas del notebook.